# Assembly Scrap Intelligence: Multi-Factor Defect Prediction via XGBoost

---
**Project 12 · LozanoLsa Industrial ML Portfolio**  
**Domain:** Electronics & Mechatronics Assembly — Scrap & Quality Control  
**Algorithm:** XGBoost (Extreme Gradient Boosting) — Binary Classification  
**Target:** `is_scrap` — Whether an assembled unit is scrapped (0 = OK · 1 = Scrap)

---

## 🏭 Business Context

In high-mix electronics and mechatronics assembly lines, scrap is the most expensive quality failure: a completed unit that cannot be reworked, recovered, or sold. Unlike dimensional defects that trigger a rework loop, scrap represents total sunk cost — materials, machine time, operator time, and downstream inspection all lost.

The challenge is that assembly scrap is **multivariate and nonlinear**. A torque deviation alone rarely creates scrap; it is the combination of under-torque, elevated line vibration, and a novice operator on a shift-change night that pushes a unit over the edge. Classical control charts, monitoring variables individually, systematically miss these interaction effects.

**XGBoost captures what control charts cannot.** As a gradient-boosted ensemble of decision trees, it models interactions, nonlinearities, and threshold effects natively — without requiring the engineer to specify them. It asks every feature jointly: *"given all current process conditions, what is the probability this unit will be scrapped?"*

This project covers 10,000 assembly cycles from a mechatronics module line, with 12 process, operator, environmental, and material features. The model achieves **ROC-AUC = 0.61** at the standard threshold — modest by clean-data standards, but reflecting a realistic signal-to-noise ratio in production assembly where scrap has genuinely stochastic components. At an adjusted threshold (0.25), recall climbs to **69%** with F1 = 0.47, making it operationally viable as a pre-dispatch risk flag.

The notebook also demonstrates why **threshold selection is a business decision, not a statistical one**: the tradeoff between catching more scrap (high recall) and generating more false alarms (low precision) depends on the relative cost of a missed defect vs. a blocked-good-unit.

---

*"When twelve variables interact to produce a defect, you need a model that thinks about all twelve simultaneously."*


## 1 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay,
    precision_recall_curve, average_precision_score,
)
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier
import shap

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

C_BLUE  = "#4C9BE8"
C_RED   = "#E8574C"
C_GREEN = "#2ECC71"
C_AMBER = "#F39C12"
C_DARK  = "#1B2840"
C_MUTED = "#697888"
C_PURP  = "#9B59B6"

FEATURES = [
    "day_of_week", "applied_torque_nm", "screwdriving_speed_rpm",
    "motor_current_a", "press_pressure_bar", "operator_cycle_time_s",
    "relative_humidity_pct", "shop_floor_temp_c", "line_vibration_mm_s",
    "operator_experience_yrs", "shift_change", "material_batch",
]
TARGET   = "is_scrap"

# Operational threshold — recall-optimised for scrap detection
THRESHOLD = 0.25   # lower than 0.5 to prioritise catching scrap

print(f"✓ Environment ready — {len(FEATURES)} features  |  target: {TARGET}")
print(f"  Operational threshold: {THRESHOLD} (recall-optimised)")


## 2 · Load Data

The dataset covers 10,000 assembly cycles from a mechatronics module line over a 60-day production window. Records were captured from the torque controller PLC, press sensor log, environmental monitoring system, and operator MES entries.

| Column | Type | Description |
|---|---|---|
| `timestamp` | datetime | Production timestamp |
| `day_of_week` | int | 0 = Monday … 6 = Sunday |
| `applied_torque_nm` | float | Torque applied to fasteners (Nm) |
| `screwdriving_speed_rpm` | float | Screw driver rotation speed (rpm) |
| `motor_current_a` | float | Driver motor current draw (A) |
| `press_pressure_bar` | float | Press station pressure (bar) |
| `operator_cycle_time_s` | float | Operator task cycle time (s) |
| `relative_humidity_pct` | float | Ambient humidity in assembly bay (%) |
| `shop_floor_temp_c` | float | Ambient temperature (°C) |
| `line_vibration_mm_s` | float | Assembly line vibration amplitude (mm/s) |
| `operator_experience_yrs` | int | Years of experience for the assigned operator |
| `shift_change` | int | 1 = cycle occurs during shift changeover window |
| `material_batch` | int | Material batch ID (1–50) |
| `is_scrap` | int | **Target** — 0 = OK · 1 = Scrap |


In [ ]:
try:
    df = pd.read_csv("assy_scrap_data.csv", parse_dates=["timestamp"])
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/LozanoLsa/XGBoost_Advanced_Scrap_Prediction/main/assy_scrap_data.csv", parse_dates=["timestamp"])

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range   : {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
print(f"Scrap rate   : {df[TARGET].mean()*100:.2f}%")
df.head()

## 3 · Sanity Checks

In [ ]:
print("── Shape ────────────────────────────")
print(f"  Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

print("\n── Missing / Duplicates ────────────")
print(f"  Missing : {df.isna().sum().sum()}" if df.isna().any().any() else "  ✓ No missing values.")
print(f"  Duplicates: {df.duplicated().sum()}" if df.duplicated().any() else "  ✓ No duplicates.")

print("\n── Data Types ──────────────────────")
print(df[FEATURES + [TARGET]].dtypes.to_string())

print("\n── Target Distribution ─────────────")
vc = df[TARGET].value_counts()
print(f"  OK    (0): {vc[0]:,} ({vc[0]/len(df)*100:.1f}%)")
print(f"  Scrap (1): {vc[1]:,} ({vc[1]/len(df)*100:.1f}%)")
print(f"  Class imbalance ratio: {vc[0]/vc[1]:.2f}:1")
print(f"\n── Descriptive Statistics ──────────")
display(df[FEATURES].describe().round(2).T)


In [ ]:
print("── Scrap Rate by Shift Change ──────")
for v, label in [(0,"Normal shift"), (1,"Shift changeover")]:
    sub = df[df["shift_change"]==v]
    print(f"  {label}: {sub[TARGET].mean()*100:.1f}% scrap ({len(sub):,} records)")

print("\n── Scrap Rate by Vibration Band ────")
df["vib_band"] = pd.cut(df["line_vibration_mm_s"],[0,1.5,3.0,3.5,6.0],
                         labels=["Low","Moderate","High","Very High"])
for band in ["Low","Moderate","High","Very High"]:
    sub = df[df["vib_band"]==band]
    if len(sub) > 0:
        print(f"  {band:10}: {sub[TARGET].mean()*100:.1f}% scrap ({len(sub):,} records)")

print("\n── Scrap Rate by Experience Level ──")
df["exp_band"] = pd.cut(df["operator_experience_yrs"],[-1,1,4,9,15],
                          labels=["Novice(0-1y)","Junior(2-4y)","Mid(5-9y)","Senior(10y+)"])
for band in ["Novice(0-1y)","Junior(2-4y)","Mid(5-9y)","Senior(10y+)"]:
    sub = df[df["exp_band"]==band]
    print(f"  {band:15}: {sub[TARGET].mean()*100:.1f}% scrap ({len(sub):,} records)")


## 4 · Exploratory Data Analysis

Assembly scrap is the result of concurrent risk accumulation. A single variable rarely tells the full story — the interesting signal is in the **joint distribution** of process, operator, and environmental conditions. EDA here serves three purposes:

1. Understand the marginal scrap rates across key risk factors
2. Identify which variables show the clearest scrap signal individually
3. Establish the visual case for why a nonlinear, interaction-capturing model like XGBoost is appropriate


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── (A) Target distribution
ax = axes[0]
counts = df[TARGET].value_counts()
bars = ax.bar(["OK (0)", "Scrap (1)"], counts.values,
              color=[C_BLUE, C_RED], alpha=0.80, edgecolor="white", width=0.55)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
            f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Count", fontsize=11); ax.set_title("Class Distribution", fontsize=12, fontweight="bold")

# ── (B) Scrap rate by vibration band
ax2 = axes[1]
scrap_by_vib = df.groupby("vib_band", observed=True)[TARGET].mean() * 100
bar_c = [C_GREEN if v < 30 else C_AMBER if v < 40 else C_RED for v in scrap_by_vib.values]
ax2.bar(scrap_by_vib.index, scrap_by_vib.values, color=bar_c, alpha=0.80, edgecolor="white")
ax2.axhline(df[TARGET].mean()*100, color=C_MUTED, ls="--", lw=1.5,
            label=f"Overall avg {df[TARGET].mean()*100:.1f}%")
ax2.set_xlabel("Vibration Band", fontsize=10); ax2.set_ylabel("Scrap Rate (%)", fontsize=10)
ax2.set_title("Scrap Rate by Vibration Level", fontsize=12, fontweight="bold")
ax2.legend(fontsize=8)

# ── (C) Scrap rate by day of week
ax3 = axes[2]
day_labels = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
scrap_by_day = df.groupby("day_of_week")[TARGET].mean() * 100
ax3.bar(day_labels[:len(scrap_by_day)], scrap_by_day.values, color=C_PURP, alpha=0.75, edgecolor="white")
ax3.axhline(df[TARGET].mean()*100, color=C_MUTED, ls="--", lw=1.5)
ax3.set_xlabel("Day of Week", fontsize=10); ax3.set_ylabel("Scrap Rate (%)", fontsize=10)
ax3.set_title("Scrap Rate by Day of Week", fontsize=12, fontweight="bold")

plt.suptitle("Assembly Scrap — Target Overview & Key Risk Dimensions",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

key_feats = ["applied_torque_nm", "line_vibration_mm_s", "motor_current_a",
             "operator_experience_yrs", "relative_humidity_pct", "operator_cycle_time_s"]

for i, feat in enumerate(key_feats):
    ax = axes[i]
    ok    = df[df[TARGET]==0][feat]
    scrap = df[df[TARGET]==1][feat]
    bins  = np.linspace(df[feat].min(), df[feat].max(), 35)
    ax.hist(ok,    bins=bins, alpha=0.60, color=C_BLUE,  density=True, label="OK")
    ax.hist(scrap, bins=bins, alpha=0.60, color=C_RED,   density=True, label="Scrap")
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.set_title(feat, fontsize=10, fontweight="bold")
    ax.legend(fontsize=7)

plt.suptitle("Feature Distributions: OK vs Scrap — Nonlinear Separation Visible",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

print("Note: overlapping distributions confirm that no single variable perfectly separates classes.")
print("XGBoost captures the JOINT effect of all features simultaneously.")


In [ ]:
# ── Temporal scrap pattern ───────────────────────────────────────────
df["date"] = df["timestamp"].dt.date
daily_scrap = df.groupby("date")[TARGET].mean() * 100

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(range(len(daily_scrap)), daily_scrap.values, color=C_BLUE, lw=1.2, alpha=0.7)
ax.fill_between(range(len(daily_scrap)), daily_scrap.values,
                alpha=0.15, color=C_BLUE)
ax.axhline(daily_scrap.mean(), color=C_RED, ls="--", lw=1.5,
           label=f"Mean = {daily_scrap.mean():.1f}%")
ax.set_xlabel("Production Day (chronological)", fontsize=11)
ax.set_ylabel("Daily Scrap Rate (%)", fontsize=11)
ax.set_title("Scrap Rate Over 60-Day Production Window\n"
             "Spikes correlate with shift changes, critical material batches, and humidity events",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


## 5 · Preprocessing

XGBoost does not require feature scaling — gradient boosted trees are invariant to monotone transformations of input features. The preprocessing steps are:

1. **Drop non-predictive columns** — `timestamp` is excluded (the temporal features `day_of_week` and `shift_change` already capture calendar signal)
2. **Stratified train/test split** — ensures both sets maintain the 30.3% scrap rate
3. **Class imbalance handling** — with a 2.3:1 OK:Scrap ratio, the default XGBoost model will be biased toward predicting OK. We address this via **threshold adjustment** at prediction time rather than oversampling, preserving the real class distribution in training.

**Threshold philosophy:** The standard threshold of 0.5 maximises accuracy but minimises recall for the minority class. Since a missed scrap (false negative) is more costly than a false alarm (false positive) in most assembly operations, we lower the threshold to 0.25. This doubles recall from 16% to 69% at the cost of precision dropping from 51% to 35% — a tradeoff each operation must price explicitly.


In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]:,} rows  |  Scrap: {y_train.mean()*100:.2f}%")
print(f"Test : {X_test.shape[0]:,} rows  |  Scrap: {y_test.mean()*100:.2f}%")
print(f"\nClass ratio (OK:Scrap) = {(y_train==0).sum()/(y_train==1).sum():.2f}:1")
print(f"\nXGBoost note: no feature scaling required — trees are scale-invariant")
print(f"Threshold strategy: use {THRESHOLD} instead of 0.5 to prioritise recall")


## 6 · Model Training

### Why XGBoost for assembly scrap prediction?

**Logistic regression** models a linear boundary — it cannot capture the critical threshold effects in assembly (e.g., scrap risk jumps nonlinearly when vibration crosses 3.5 mm/s, not gradually). **Random Forest** is non-parametric but does not learn from errors iteratively. **XGBoost** builds trees sequentially, each correcting the residual errors of the previous — it is specifically optimised to find the difficult boundary cases that earlier trees missed.

The sequential correction mechanism is what makes XGBoost effective on production data:
- It finds the samples with the highest prediction error (the ambiguous cases near the decision boundary)
- Each new tree is weighted to focus on those samples
- After 200 iterations, the ensemble has specialised attention across the full risk landscape

**Key hyperparameters:**
- `n_estimators = 200` — number of sequential trees
- `max_depth = 4` — maximum tree depth (limits overfitting)
- `learning_rate = 0.1` — shrinkage applied to each tree's contribution
- `subsample = 0.8` — row sampling per tree (reduces overfitting)
- `colsample_bytree = 0.8` — feature sampling per tree (further regularisation)


In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

xgb.fit(X_train, y_train)

print("✓ XGBoost fitted")
print(f"  Trees built: {xgb.n_estimators}")
print(f"  Features used: {len(FEATURES)}")
print(f"  Training complete — generating predictions at two thresholds:")
print(f"    Default (0.50): maximises accuracy")
print(f"    Recall-opt ({THRESHOLD}): maximises scrap detection recall")


## 7 · Model Evaluation

| Metric | Default (thr=0.50) | Recall-Opt (thr=0.25) | Operational Meaning |
|---|---|---|---|
| **ROC-AUC** | **0.610** | 0.610 | Threshold-independent discrimination |
| **Recall** | 16% | **69%** | % of actual scrap units caught |
| **Precision** | 51% | 35% | % of flagged units that are truly scrap |
| **F1** | 0.24 | **0.47** | Harmonic mean — favours the recall-optimised threshold |
| **Accuracy** | 70% | 58% | Not the primary metric for imbalanced scrap data |

**ROC-AUC = 0.61** reflects a realistic assembly dataset where scrap has genuine stochastic components (a unit can fail even under ideal conditions). The practical threshold for operations is 0.25 — this catches nearly 7 in 10 scrap units before they leave the line.


In [ ]:
y_proba_test  = xgb.predict_proba(X_test)[:, 1]
y_proba_train = xgb.predict_proba(X_train)[:, 1]

y_pred_50 = xgb.predict(X_test)                          # default threshold
y_pred_25 = (y_proba_test >= THRESHOLD).astype(int)      # recall-optimised

auc_test = roc_auc_score(y_test, y_proba_test)

print(f"XGBoost — Performance Summary")
print(f"  ROC-AUC (test) : {auc_test:.4f}")
print(f"\n  ─── Default threshold = 0.50 ───────────────────────")
print(f"  Accuracy  : {accuracy_score(y_test,y_pred_50):.4f}")
print(f"  Precision : {precision_score(y_test,y_pred_50):.4f}")
print(f"  Recall    : {recall_score(y_test,y_pred_50):.4f}")
print(f"  F1        : {f1_score(y_test,y_pred_50):.4f}")
print(f"\n  ─── Operational threshold = {THRESHOLD} ────────────────")
print(f"  Accuracy  : {accuracy_score(y_test,y_pred_25):.4f}")
print(f"  Precision : {precision_score(y_test,y_pred_25):.4f}")
print(f"  Recall    : {recall_score(y_test,y_pred_25):.4f}")
print(f"  F1        : {f1_score(y_test,y_pred_25):.4f}")
print(f"\nClassification Report (threshold = {THRESHOLD}):")
print(classification_report(y_test, y_pred_25, target_names=["OK","Scrap"]))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── (A) Confusion matrix (operational threshold)
ax = axes[0]
cm = confusion_matrix(y_test, y_pred_25)
tn, fp, fn, tp = cm.ravel()
cm_labels = np.array([[f"TN\n{tn}", f"FP\n{fp}"],
                       [f"FN\n{fn}", f"TP\n{tp}"]])
colors = np.array([[C_GREEN, C_AMBER], [C_RED, C_BLUE]])
for i in range(2):
    for j in range(2):
        ax.add_patch(plt.Rectangle((j, 1-i), 1, 1,
                     facecolor=colors[i,j], alpha=0.65, edgecolor="white", lw=2))
        ax.text(j+0.5, 1.5-i, cm_labels[i,j], ha="center", va="center",
                fontsize=13, fontweight="bold", color="white")
ax.set_xlim(0,2); ax.set_ylim(0,2)
ax.set_xticks([0.5,1.5]); ax.set_yticks([0.5,1.5])
ax.set_xticklabels(["Pred: OK","Pred: Scrap"], fontsize=10)
ax.set_yticklabels(["Actual: Scrap","Actual: OK"], fontsize=10)
ax.set_title(f"Confusion Matrix (threshold={THRESHOLD})", fontsize=11, fontweight="bold")
ax.text(0.5, -0.12, f"Missed scrap (FN): {fn}  |  False alarms (FP): {fp}",
        transform=ax.transAxes, ha="center", fontsize=8, color=C_MUTED)

# ── (B) ROC Curve
ax2 = axes[1]
RocCurveDisplay.from_estimator(xgb, X_test, y_test, ax=ax2, color=C_BLUE,
                                name=f"XGBoost (AUC={auc_test:.3f})")
ax2.plot([0,1],[0,1], color=C_MUTED, ls="--", lw=1.2, label="Random")
ax2.set_title("ROC Curve — Test Set", fontsize=12, fontweight="bold")
ax2.legend(fontsize=8)

# ── (C) Precision-Recall Curve
ax3 = axes[2]
prec_arr, rec_arr, thr_arr = precision_recall_curve(y_test, y_proba_test)
ap = average_precision_score(y_test, y_proba_test)
ax3.plot(rec_arr, prec_arr, color=C_RED, lw=1.5, label=f"XGBoost (AP={ap:.3f})")
ax3.axhline(y_test.mean(), color=C_MUTED, ls="--", lw=1.2, label=f"Baseline ({y_test.mean():.2f})")
# Mark operational threshold
idx = np.argmin(np.abs(thr_arr - THRESHOLD))
ax3.scatter([rec_arr[idx]], [prec_arr[idx]], s=80, color=C_AMBER, zorder=5,
            label=f"threshold={THRESHOLD}")
ax3.set_xlabel("Recall", fontsize=11); ax3.set_ylabel("Precision", fontsize=11)
ax3.set_title("Precision-Recall Curve", fontsize=12, fontweight="bold")
ax3.legend(fontsize=8)

plt.tight_layout(); plt.show()

print(f"At threshold={THRESHOLD}: catching {tp}/{tp+fn} ({tp/(tp+fn)*100:.0f}%) scrap units")
print(f"  At cost of {fp} false alarms out of {tn+fp} truly-good units ({fp/(tn+fp)*100:.1f}%)")


## 8 · Interpretability

XGBoost provides two complementary interpretability tools:

1. **Built-in feature importance (gain)** — the average gain in prediction accuracy contributed by splits on each feature across all trees. Fast but does not account for feature interactions or directionality.

2. **SHAP (SHapley Additive exPlanations)** — a game-theoretic framework that assigns each feature a contribution to each individual prediction. SHAP values are locally faithful (they explain specific predictions, not just global averages), show direction (positive = increases scrap probability), and respect feature interactions. This is the gold standard for tree-based model interpretability.

The SHAP summary plot shows both **magnitude** (x-axis position) and **direction** of each feature's effect, colored by the feature's actual value — making it possible to answer questions like "does *high* vibration increase or decrease scrap probability?"


In [ ]:
# ── Built-in XGBoost Feature Importance ──────────────────────────────
imp_df = pd.DataFrame({
    "Feature"   : FEATURES,
    "Importance": xgb.feature_importances_,
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
bar_c = [C_RED if v > imp_df["Importance"].quantile(0.75) else
         C_AMBER if v > imp_df["Importance"].quantile(0.5) else C_BLUE
         for v in imp_df["Importance"]]
bars = ax.barh(imp_df["Feature"], imp_df["Importance"],
               color=bar_c, alpha=0.82, edgecolor="none", height=0.65)
for bar, val in zip(bars, imp_df["Importance"]):
    ax.text(val+0.001, bar.get_y()+bar.get_height()/2, f"{val:.4f}",
            va="center", fontsize=9, color=C_DARK)
ax.set_xlabel("XGBoost Feature Importance (Gain)", fontsize=11)
ax.set_title("Built-in Feature Importance — Information Gained per Split\n"
             "Red = top quartile · Amber = second · Blue = bottom half",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

print("Top 5 features by XGBoost gain:")
for _, row in imp_df.iloc[::-1].head(5).iterrows():
    print(f"  {row['Feature']:<30}: {row['Importance']:.4f}")


In [ ]:
# ── SHAP — Game-theoretic feature attribution ────────────────────────
shap.initjs()

explainer   = shap.TreeExplainer(xgb)
X_sample    = X_train.sample(n=1000, random_state=42)
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES,
                  plot_type="dot", show=False, max_display=12)
plt.title("SHAP Summary Plot — Direction & Magnitude of Each Feature's Effect\n"
          "Color = feature value (red=high, blue=low) · x-axis = impact on scrap probability",
          fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

print("How to read this plot:")
print("  → Each dot = one sample from the training set")
print("  → Positive x: feature increases P(scrap)")
print("  → Negative x: feature decreases P(scrap)")
print("  → Red dot: sample has HIGH value of that feature")
print("  → Blue dot: sample has LOW value of that feature")


In [ ]:
# ── SHAP Force Plot — single prediction explanation ──────────────────
# Pick the highest-risk sample from the test set
high_risk_idx = y_proba_test.argmax()
shap_single = explainer.shap_values(X_test.iloc[[high_risk_idx]])

print(f"Explaining highest-risk test sample (index {high_risk_idx}):")
print(f"  Predicted scrap probability: {y_proba_test[high_risk_idx]:.2%}")
print(f"  Actual label              : {'Scrap' if y_test.iloc[high_risk_idx]==1 else 'OK'}")
print(f"\nParameter values for this sample:")
for feat in FEATURES:
    val = X_test.iloc[high_risk_idx][feat]
    print(f"  {feat:<30}: {val}")

plt.figure()
shap.force_plot(
    explainer.expected_value,
    shap_single[0],
    X_test.iloc[high_risk_idx],
    feature_names=FEATURES,
    matplotlib=True,
    show=False,
)
plt.title(f"SHAP Force Plot — Highest-Risk Sample (P(scrap)={y_proba_test[high_risk_idx]:.2%})",
          fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()


## 9 · Statistical Validation

For a classification model, the relevant diagnostics are:

- **Probability calibration** — are predicted probabilities reliable? (A 70% predicted probability should correspond to ~70% actual scrap rate)
- **Score distributions** — do OK and Scrap units show clearly different probability distributions?
- **Permutation importance** — confirms that feature importances are genuine, not artefacts of the specific training data split


In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── (A) Calibration curve
ax = axes[0]
fraction_pos, mean_pred = calibration_curve(y_test, y_proba_test, n_bins=10)
ax.plot(mean_pred, fraction_pos, marker="o", color=C_BLUE, lw=1.5, label="XGBoost")
ax.plot([0,1],[0,1], color=C_MUTED, ls="--", lw=1.2, label="Perfect calibration")
ax.set_xlabel("Mean Predicted Probability", fontsize=10)
ax.set_ylabel("Fraction of Positives (actual)", fontsize=10)
ax.set_title("Probability Calibration Curve\nAre predicted probabilities reliable?",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=8)

# ── (B) Score distributions by class
ax2 = axes[1]
ok_scores    = y_proba_test[y_test.values==0]
scrap_scores = y_proba_test[y_test.values==1]
ax2.hist(ok_scores,    bins=30, alpha=0.60, color=C_BLUE,  density=True, label="Actual OK")
ax2.hist(scrap_scores, bins=30, alpha=0.60, color=C_RED,   density=True, label="Actual Scrap")
ax2.axvline(THRESHOLD, color=C_AMBER, lw=1.8, ls="--",
            label=f"Threshold = {THRESHOLD}")
ax2.set_xlabel("Predicted P(Scrap)", fontsize=10); ax2.set_ylabel("Density", fontsize=10)
ax2.set_title("Score Distributions by True Label\nSeparation confirms model learns signal",
              fontsize=11, fontweight="bold")
ax2.legend(fontsize=8)

# ── (C) Permutation importance (model-agnostic)
ax3 = axes[2]
pi = permutation_importance(xgb, X_test, y_test, n_repeats=15,
                             random_state=42, scoring="roc_auc")
pi_df = pd.DataFrame({"Feature":FEATURES, "Mean":pi.importances_mean,
                       "Std":pi.importances_std})          .sort_values("Mean", ascending=True)
bar_c3 = [C_RED if v>0.005 else C_AMBER if v>0.001 else C_MUTED for v in pi_df["Mean"]]
ax3.barh(pi_df["Feature"], pi_df["Mean"], xerr=pi_df["Std"],
         color=bar_c3, alpha=0.82, edgecolor="none", height=0.60, capsize=3)
ax3.set_xlabel("Mean ROC-AUC drop when permuted", fontsize=9)
ax3.set_title("Permutation Importance (AUC drop)\nConfirms genuine signal vs chance",
              fontsize=11, fontweight="bold")

plt.tight_layout(); plt.show()

print("Calibration note:")
print("  Points close to diagonal = reliable probability estimates")
print("  XGBoost often needs calibration for extreme probabilities")


## 10 · Process Simulator

Three scenarios demonstrate the model's most important operational insight: **process risk and human/environmental risk are additive, but process control provides the fastest relief**.

| Scenario | Story | Expected |
|---|---|---|
| **A — Standard, controlled conditions** | Experienced operator, normal process, no vibration | Low risk |
| **B — All risk factors active simultaneously** | Novice + shift change + vibration + humidity spike | Maximum risk |
| **C — Process corrected, human risk remains** | Same operator/shift as B, but torque and vibration corrected | Partial recovery |


In [ ]:
MEDIAN_VALUES = X_train.mean(numeric_only=True).to_dict()

def predict_scrap_risk(threshold=THRESHOLD, **kwargs):
    """
    Predict scrap probability for a given set of process conditions.
    Unspecified features default to training set means.

    Returns
    -------
    prob_scrap : float — predicted scrap probability (0–1)
    decision   : str   — 'Flag as SCRAP RISK' or 'OK to proceed'
    """
    sample = MEDIAN_VALUES.copy()
    sample.update(kwargs)
    x = pd.DataFrame([[sample[c] for c in FEATURES]], columns=FEATURES)
    prob = xgb.predict_proba(x)[0, 1]
    decision = "⚠ Flag as SCRAP RISK" if prob >= threshold else "✅ OK to proceed"
    return prob, decision


In [ ]:
# ══ Scenario A — Standard, controlled ════════════════════════════════
a_prob, a_dec = predict_scrap_risk(
    applied_torque_nm=1.85, motor_current_a=1.8, line_vibration_mm_s=1.0,
    press_pressure_bar=5.0, operator_experience_yrs=5, shift_change=0,
    relative_humidity_pct=40, shop_floor_temp_c=24, operator_cycle_time_s=18
)
print("═══ Scenario A — Controlled Conditions ══════════════════════════")
print(f"  Torque: 1.85 Nm | Vibration: 1.0 mm/s | Exp: 5y | Humidity: 40% | Shift: normal")
print(f"  → P(Scrap): {a_prob:.2%}   {a_dec}")

# ══ Scenario B — All risk factors active ═════════════════════════════
b_prob, b_dec = predict_scrap_risk(
    applied_torque_nm=1.4, motor_current_a=2.5, line_vibration_mm_s=4.8,
    press_pressure_bar=4.2, operator_experience_yrs=0, shift_change=1,
    relative_humidity_pct=70, shop_floor_temp_c=29, operator_cycle_time_s=26
)
print("\n═══ Scenario B — All Risk Factors Active ════════════════════════")
print(f"  Torque: 1.4 Nm | Vibration: 4.8 mm/s | Exp: 0y | Humidity: 70% | Shift: changeover")
print(f"  → P(Scrap): {b_prob:.2%}   {b_dec}")
print(f"  ⚠ Compounding risks: low torque + high vibration + novice operator + night shift.")

# ══ Scenario C — Process corrected, human factors remain ══════════════
c_prob, c_dec = predict_scrap_risk(
    applied_torque_nm=1.85, motor_current_a=1.8, line_vibration_mm_s=1.5,
    press_pressure_bar=5.0, operator_experience_yrs=0, shift_change=1,
    relative_humidity_pct=70, shop_floor_temp_c=29, operator_cycle_time_s=20
)
print("\n═══ Scenario C — Process Corrected · Human Risks Remain ══════════")
print(f"  Torque: 1.85 Nm | Vibration: 1.5 mm/s | Exp: 0y | Humidity: 70% | Shift: changeover")
print(f"  → P(Scrap): {c_prob:.2%}   {c_dec}")
print(f"\n  Correcting torque and vibration alone reduces risk by {b_prob-c_prob:.0%} percentage points.")
print(f"  Human factors (0-year operator on shift change) still contribute residual risk.")
print(f"  → Recommendation: pair novice operators with mentors during shift changes.")


In [ ]:
# ── Risk Grid: Torque × Vibration ────────────────────────────────────
torque_range = np.linspace(1.2, 2.3, 50)
vibration_range = np.linspace(0.3, 6.0, 50)
T, V = np.meshgrid(torque_range, vibration_range)

grid = pd.DataFrame([{c: MEDIAN_VALUES[c] for c in FEATURES}] * 2500)
grid["applied_torque_nm"]   = T.ravel()
grid["line_vibration_mm_s"] = V.ravel()

Z = xgb.predict_proba(grid[FEATURES])[:,1].reshape(T.shape)

fig, ax = plt.subplots(figsize=(9, 6))
cf = ax.contourf(T, V, Z, levels=25, cmap="RdYlGn_r", alpha=0.88)
cbar = plt.colorbar(cf, ax=ax); cbar.set_label("P(Scrap)")
cs = ax.contour(T, V, Z, levels=[THRESHOLD], colors=["white"], linewidths=2.0)
ax.clabel(cs, fmt=f"threshold={THRESHOLD}", fontsize=9, colors="white")
ax.contourf(T, V, Z, levels=[0, THRESHOLD], colors=["lime"], alpha=0.15, hatches=["////"])
ax.set_xlabel("Applied Torque (Nm)", fontsize=11)
ax.set_ylabel("Line Vibration (mm/s)", fontsize=11)
ax.set_title("Scrap Risk Map — Torque × Vibration\n"
             "Green zone = P(Scrap) below operational threshold\n"
             "(medians for all other features)",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()
print("Operating in the green zone for BOTH torque and vibration simultaneously")
print("is the strongest single process control action available.")


## 11 · Final Reflection

---

### What XGBoost found

Twelve variables enter the model; the SHAP analysis reveals a nuanced importance hierarchy that XGBoost's built-in gain metric partially obscures:

- **`shift_change`** is the single highest-gain feature — shift changeovers create the most information gain in the tree splits, consistent with the operational reality that the first and last cycles of a shift are statistically different from steady-state production.
- **`line_vibration_mm_s`** is the strongest continuous process driver — its SHAP values are high-magnitude and directional (high vibration → high scrap risk), visible in the distributions and confirmed by the 2D risk map.
- **`operator_experience_yrs`** shows a nonlinear threshold effect: novice operators (0–1 year) carry significantly higher risk, but the effect flattens quickly after year 2. This is not captured by any linear model.
- **`applied_torque_nm`** shows a bimodal risk pattern — too low *and* too high both increase scrap probability, a nonlinearity that XGBoost captures natively through its tree structure.

### The threshold story

The most important decision in this project is not the model architecture — it is the operating threshold. The question "should I use 0.25 or 0.50?" cannot be answered statistically. It is a **cost function problem**:

- **Cost of a missed scrap (FN):** Full material + labour cost of the unit, plus potential warranty/customer impact downstream
- **Cost of a false alarm (FP):** Manual inspection cost, possible line delay, operator fatigue from false alerts

If the assembly unit costs €500 in materials and labour, and the downstream rework/recall risk of a slipped scrap is €2,000, then the break-even precision at threshold 0.25 is 500/(500+2000) = 20% — well below the achieved 35%. At threshold 0.25, this model is economically justified.

### What the model cannot see

The model treats `material_batch` as a numerical ID (1–50). It does not know which specific batches were flagged as critical by incoming quality inspection. Encoding batch quality metadata (e.g., supplier certificate deviations, incoming inspection results) as features would likely improve precision significantly.

It also does not model the interaction between operator experience and shift type explicitly — a senior operator on a shift change may perform similarly to a junior operator on a normal shift. Feature engineering an `experience × shift_change` interaction term is a recommended next step.

---

*LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa*
